In [19]:
import pandas as pd
import os

# Define file paths
base_path = '../data/csv_data/'
files = {
    'applications': 'applications.csv',
    'cv_data': 'cv_data.csv',
    'job_data': 'job_data.csv',
    'user_data': 'user_data.csv',
    'company_data': 'company_data.csv'
}

# Load dataframes
dfs = {}
for key, filename in files.items():
    path = os.path.join(base_path, filename)
    if os.path.exists(path):
        try:
            dfs[key] = pd.read_csv(path)
            print(f"Loaded {key} with shape {dfs[key].shape}")
        except Exception as e:
            print(f"Error loading {key}: {e}")
    else:
        print(f"File not found: {path}")


Loaded applications with shape (3983, 22)
Loaded cv_data with shape (3983, 805)
Loaded job_data with shape (14634, 789)
Loaded user_data with shape (3191, 12)
Loaded company_data with shape (6511, 6)


In [20]:
# Aggregate data
if all(k in dfs for k in ['applications', 'cv_data', 'job_data']):
    df_apps = dfs['applications'].copy()
    df_cv = dfs['cv_data'].copy()
    df_jobs = dfs['job_data'].copy()

    print("\n--- Standardizing Columns ---")

    # 1. Standardize Job Data
    if 'id' in df_jobs.columns:
        print("Renaming job_data['id'] to 'job_id'")
        df_jobs = df_jobs.rename(columns={'id': 'job_id'})

    # 2. Standardize CV Data
    if 'cv_id' in df_cv.columns:
        print("Renaming cv_data['cv_id'] to 'user_id'")
        df_cv = df_cv.rename(columns={'cv_id': 'user_id'})
    elif 'id' in df_cv.columns:
        print("Renaming cv_data['id'] to 'user_id'")
        df_cv = df_cv.rename(columns={'id': 'user_id'})

    # 3. Process Applications Data (Wide to Long)
    print(f"Applications columns: {df_apps.columns.tolist()}")

    # Rename cv_id to user_id in applications if present
    if 'cv_id' in df_apps.columns:
        print("Renaming applications['cv_id'] to 'user_id'")
        df_apps = df_apps.rename(columns={'cv_id': 'user_id'})

    # Check if we need to melt (if we have job_0, job_1, etc.)
    job_cols = [c for c in df_apps.columns if c.startswith('job_') and c[4:].isdigit()]

    if job_cols:
        print(f"Detected wide format in applications with columns: {job_cols}")
        print("Melting applications to long format...")
        # Keep user_id and melt job columns
        if 'user_id' in df_apps.columns:
            df_apps_long = df_apps.melt(id_vars=['user_id'], value_vars=job_cols, value_name='job_id')
            # Drop the 'variable' column (e.g. 'job_0')
            df_apps_long = df_apps_long.drop(columns=['variable'])
            # Drop rows with missing job_ids
            df_apps_long = df_apps_long.dropna(subset=['job_id'])

            print(f"Reshaped applications shape: {df_apps_long.shape}")
            df_apps = df_apps_long
        else:
            print("Error: 'user_id' (or 'cv_id') not found in applications, cannot melt.")
    else:
        # Assume it's already in long format or has a single job column
        if 'job_id' not in df_apps.columns:
             # Try to find a single job column
             for col in ['job', 'jobId', 'job_post_id', 'id_job']:
                if col in df_apps.columns:
                    print(f"Renaming applications['{col}'] to 'job_id'")
                    df_apps = df_apps.rename(columns={col: 'job_id'})
                    break

    # 4. Merge Applications with Jobs
    print("\n--- Merging Data ---")
    # Ensure job_id types match (often one is int, one is float or object)
    if 'job_id' in df_apps.columns and 'job_id' in df_jobs.columns:
        # Convert to same type if possible
        try:
            df_apps['job_id'] = df_apps['job_id'].astype(int)
            df_jobs['job_id'] = df_jobs['job_id'].astype(int)
        except:
            pass # Keep as is if conversion fails

        merged_app_job = pd.merge(df_apps, df_jobs, on='job_id', how='inner', suffixes=('_app', '_job'))
        print(f"Merged Apps + Jobs: {merged_app_job.shape}")
    else:
        print("Error: Could not find common 'job_id' column.")
        merged_app_job = pd.DataFrame()

    # 5. Merge with CVs
    if not merged_app_job.empty:
        if 'user_id' in merged_app_job.columns and 'user_id' in df_cv.columns:
            # Convert user_id to same type
            try:
                merged_app_job['user_id'] = merged_app_job['user_id'].astype(int)
                df_cv['user_id'] = df_cv['user_id'].astype(int)
            except:
                pass

            full_data = pd.merge(merged_app_job, df_cv, on='user_id', how='inner', suffixes=('', '_cv'))
            print(f"Full merged data shape: {full_data.shape}")

            # --- Extract Target Columns ---
            final_df = pd.DataFrame()

            def get_col(df, candidates):
                for c in candidates:
                    if c in df.columns:
                        return df[c]
                    # Check suffixes
                    if f"{c}_cv" in df.columns: return df[f"{c}_cv"]
                    if f"{c}_job" in df.columns: return df[f"{c}_job"]
                return pd.Series([None] * len(df))

            # Candidate
            final_df['candidate_summary'] = get_col(full_data, ['summary', 'about', 'profile_summary', 'description_cv', 'crawled_career_objective'])
            final_df['candidate_education'] = get_col(full_data, ['education', 'degree', 'university', 'crawled_education_level'])
            final_df['candidate_skills'] = get_col(full_data, ['skills', 'skills_cv', 'candidate_skills', 'tech_stack', 'crawled_skills', 'Skills'])

            # Job
            final_df['job_post_title'] = get_col(full_data, ['title', 'job_title', 'position', 'title_job', 'Job Title'])
            final_df['job_post_description'] = get_col(full_data, ['description', 'job_description', 'description_job', 'Job Description'])
            final_df['job_post_industry'] = get_col(full_data, ['industry', 'sector', 'category', 'Industry'])
            final_df['job_post_job_type'] = get_col(full_data, ['job_type', 'employment_type', 'type', 'type_job', 'Job Type'])
            final_df['job_post_level'] = get_col(full_data, ['level', 'experience_level', 'seniority', 'Career Level'])
            final_df['job_post_skills'] = get_col(full_data, ['skills_job', 'requirements', 'job_skills', 'required_skills', 'Job Requirements'])

            print("\n--- Final Aggregated Data ---")
            display(final_df.head())

            # Save
            final_df.to_csv(os.path.join(base_path, 'aggregated_data.csv'), index=False)
            print("Saved to aggregated_data.csv")

        else:
             print(f"Error: Could not find common 'user_id' column.")
    else:
        print("Skipping CV merge due to previous error.")
else:
    print("Missing necessary dataframes.")


--- Standardizing Columns ---
Renaming cv_data['cv_id'] to 'user_id'
Applications columns: ['application_id', 'cv_id', 'job_0', 'dist_0', 'job_1', 'dist_1', 'job_2', 'dist_2', 'job_3', 'dist_3', 'job_4', 'dist_4', 'job_5', 'dist_5', 'job_6', 'dist_6', 'job_7', 'dist_7', 'job_8', 'dist_8', 'job_9', 'dist_9']
Renaming applications['cv_id'] to 'user_id'
Detected wide format in applications with columns: ['job_0', 'job_1', 'job_2', 'job_3', 'job_4', 'job_5', 'job_6', 'job_7', 'job_8', 'job_9']
Melting applications to long format...
Reshaped applications shape: (39830, 2)

--- Merging Data ---
Merged Apps + Jobs: (39830, 790)
Full merged data shape: (39830, 1594)

--- Final Aggregated Data ---


,candidate_summary,candidate_education,candidate_skills,job_post_title,job_post_description,job_post_industry,job_post_job_type,job_post_level,job_post_skills
0,Với kinh nghiệm 10 năm trong lĩnh vực lái xe t...,2.0,"- Bằng lái xe hạng C, D, E và có kinh nghiệm 5...",Nhân Viên Lái Xe Văn Phòng (Bằng B2),Lái xe văn phòng chở CBNV và lãnh đạo theo yêu...,Vận tải / Kho vận,Full time,Nhân viên,Bằng cấp lái xe B2;Kinh nghiệm lái xe văn phòn...
1,Gắn bó lâu dài với công việc cũng như doanh ng...,2.0,Có chí tiến thủ đối với công việc\r\nCó khả nă...,"Nhân viên kinh doanh, chăm sóc khách hàng VẬN ...",CÔNG TY TNHH THÁI KHÔN CẦN TUYỂN DỤNG Nhân viê...,nhân viên kinh doanh vận tải,Full time,Nhân viên,"- Cẩn thận, chăm chỉ. - Có trách nhiệm trong c..."
2,- Tìm kiếm một vị trí nhân viên hành chính và ...,0.0,- Am hiểu mọi quy trình về hành chính nhân sự ...,Quản lý nhân sự,- Quản lý các công việc của bộ phận nhân sự - ...,nhân sự tiếng Trung,Full time,Quản lý cấp cao,-Tiếng HOa lưu loát 4 kỹ năng - Tốt nghiệp đại...
3,Trở thành kỹ sư tốt trong công tác sửa chữa xe...,0.0,"Có khả năng chu toàn công việc, siêng năng, tỉ...",Nhân Viên Lái Xe,"Lái xe phục vụ cho Tổng giám đốc (CEO), theo đ...",Ngành nghề,Full time,Nhân viên,"Giới tính: Nam, dưới 35 tuổi, sức khỏe tốt.Có ..."
4,NaN,NaN,NaN,Nhân Viên Kinh Doanh Nội Thất,"- Trực Fanpage, trả lời tin nhắn, tư vấn và ch...",Kinh doanh / Bán hàng,Full time,Nhân viên,"- Xử lý linh hoạt, nói chuyện dễ nghe- Biết sử..."


Saved to aggregated_data.csv
